# Módulo 7 · Práctica: optimizadores en el valle de Rosenbrock

Vas a programar desde cero, con NumPy, los optimizadores que entrenan a los LLM: descenso de gradiente, SGD con minibatches, momentum, RMSProp, Adam y AdamW. Después los soltarás en la función de Rosenbrock, un valle curvo y estrecho que castiga a los métodos ingenuos, y dibujarás sus trayectorias.

**Cómo trabajar:** ejecuta las celdas en orden. Completa las celdas `# TU CÓDIGO` y ejecuta la `# Comprobación` que va detrás.

| Parte | Qué haces |
| --- | --- |
| 0 | La función de Rosenbrock y su gradiente |
| 1 | Descenso de gradiente y número de condición |
| 2 | SGD: el minibatch como estimador insesgado |
| 3 | Momentum, RMSProp y Adam |
| 4 | Carrera en el valle: trayectorias y curvas de pérdida |
| 5 | Warmup y cosine decay |
| 6 | AdamW: weight decay desacoplado |
| 7 | Retos |

## Parte 0 · La función de Rosenbrock

$$f(x, y) = (1 - x)^2 + 100\,(y - x^2)^2$$

Su mínimo está en $(1, 1)$ con $f = 0$. El fondo del valle sigue la parábola $y = x^2$: bajar al valle es fácil, recorrerlo es difícil.

**Ejercicio 1.** Implementa el gradiente $\nabla f = \big(-2(1-x) - 400x(y - x^2),\; 200(y - x^2)\big)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rosen(p):
    x, y = p
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosen_grad(p):
    x, y = p
    # TU CÓDIGO: devuelve np.array([df/dx, df/dy])
    return None

In [ ]:
# Comprobación: gradiente analítico frente a diferencias centradas (módulo 4)
def grad_num(f, p, h=1e-6):
    p = np.asarray(p, float)
    return np.array([(f(p + h*e) - f(p - h*e)) / (2*h) for e in np.eye(len(p))])

for p in [(-1.5, 2.0), (0.3, -0.7), (1.0, 1.0)]:
    g = rosen_grad(np.array(p))
    assert g is not None and np.allclose(g, grad_num(rosen, p), atol=1e-4), f"Gradiente incorrecto en {p}"
assert np.allclose(rosen_grad(np.array([-1.5, 2.0])), [-155, -50]), "En (-1,5; 2) debe dar (-155, -50)"
print("Correcto. f(-1,5; 2) =", rosen((-1.5, 2.0)))

## Parte 1 · Descenso de gradiente y número de condición

Regla: $\theta_{k+1} = \theta_k - \eta\,\nabla f(\theta_k)$. En una cuadrática $f = \tfrac12\theta^\top H\theta$ el error en cada autodirección se multiplica por $(1 - \eta\lambda_i)$. Con el mejor $\eta = 2/(\lambda_{\min}+\lambda_{\max})$ el factor es $(\kappa-1)/(\kappa+1)$, con $\kappa = \lambda_{\max}/\lambda_{\min}$.

**Ejercicio 2.** Escribe `gd_iters(H, eta, tol)`: aplica descenso de gradiente desde $\theta_0 = (1, 1)$ y devuelve cuántas iteraciones tarda en cumplir $\lVert\theta\rVert < \text{tol}\cdot\lVert\theta_0\rVert$ (máximo 100.000).

In [ ]:
def gd_iters(H, eta, tol=1e-3, max_it=100_000):
    theta = np.array([1.0, 1.0])
    n0 = np.linalg.norm(theta)
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación
for kappa, esperado in [(1, 1), (10, 35), (100, 346)]:
    H = np.diag([1.0, kappa])
    n = gd_iters(H, 2 / (1 + kappa))
    assert n == esperado, f"kappa = {kappa}: esperaba {esperado} iteraciones y salen {n}"
print("Correcto: con kappa = 100 hacen falta 346 iteraciones; con kappa = 1, una.")

**Pregunta.** En el mínimo de Rosenbrock el hessiano tiene autovalores ≈ 0,4 y ≈ 1.001,6. ¿Cuánto vale κ? ¿Qué tasa de aprendizaje máxima tolera el descenso de gradiente cerca del mínimo?

In [ ]:
H_min = np.array([[802., -400.], [-400., 200.]])
lam = np.linalg.eigvalsh(H_min)
print("autovalores:", lam.round(4), " kappa =", round(lam[1] / lam[0], 1), " eta_max = 2/lambda_max =", round(2 / lam[1], 5))

## Parte 2 · SGD: el minibatch como estimador insesgado

Un LLM no calcula el gradiente sobre todo el corpus: usa un minibatch de $B$ ejemplos. Su media es un estimador **insesgado** del gradiente completo, con varianza $\approx \sigma^2/B$.

Datos sintéticos: $y = 3x + 1 + \text{ruido}$, pérdida cuadrática media sobre $\theta = (w, b)$.

**Ejercicio 3.** Implementa `grad_mb(theta, idx)`, el gradiente de $\tfrac{1}{|idx|}\sum_{i \in idx} \tfrac12 (w x_i + b - y_i)^2$.

In [ ]:
rng = np.random.default_rng(0)
N = 1000
X = rng.normal(size=N)
Y = 3 * X + 1 + 0.5 * rng.normal(size=N)

def grad_mb(theta, idx):
    w, b = theta
    # TU CÓDIGO: residuo r = w x + b - y en los índices idx; devuelve (media de r·x, media de r)
    return None

theta = np.array([0.0, 0.0])
g_full = grad_mb(theta, np.arange(N))
print("gradiente completo:", g_full)

In [ ]:
# Comprobación: media de muchos minibatches ≈ gradiente completo; varianza ∝ 1/B
def muestras(B, reps=4000):
    r = np.random.default_rng(1)
    return np.array([grad_mb(theta, r.choice(N, B, replace=False)) for _ in range(reps)])

G10, G100 = muestras(10), muestras(100)
assert np.allclose(G10.mean(0), g_full, atol=0.1), "La media de los minibatches debe coincidir con el gradiente completo"
ratio = G10.var(0).sum() / G100.var(0).sum()
assert 7 < ratio < 15, f"La varianza con B = 10 debería ser unas 10 veces la de B = 100 (sale {ratio:.1f})"
print(f"Correcto. Varianza(B=10) / Varianza(B=100) = {ratio:.1f}")

## Parte 3 · Momentum, RMSProp y Adam

Todos los optimizadores tendrán la misma firma: `paso(theta, g, estado, lr)` devuelve el nuevo `theta` y guarda en el diccionario `estado` lo que necesite recordar.

$$\text{Momentum: } v \leftarrow \beta v + g,\quad \theta \leftarrow \theta - \eta v$$
$$\text{RMSProp: } s \leftarrow \rho s + (1-\rho) g^2,\quad \theta \leftarrow \theta - \eta\, g/(\sqrt{s}+\varepsilon)$$
$$\text{Adam: } m \leftarrow \beta_1 m + (1-\beta_1) g,\; v \leftarrow \beta_2 v + (1-\beta_2) g^2,\; \hat m = \frac{m}{1-\beta_1^t},\; \hat v = \frac{v}{1-\beta_2^t},\; \theta \leftarrow \theta - \eta\,\frac{\hat m}{\sqrt{\hat v}+\varepsilon}$$

**Ejercicio 4.** Completa los tres pasos.

In [ ]:
def paso_gd(theta, g, estado, lr):
    return theta - lr * g

def paso_momentum(theta, g, estado, lr, beta=0.9):
    v = estado.get("v", np.zeros_like(theta))
    # TU CÓDIGO
    return None

def paso_rmsprop(theta, g, estado, lr, rho=0.9, eps=1e-8):
    s = estado.get("s", np.zeros_like(theta))
    # TU CÓDIGO
    return None

def paso_adam(theta, g, estado, lr, b1=0.9, b2=0.999, eps=1e-8):
    t = estado.get("t", 0) + 1
    m = estado.get("m", np.zeros_like(theta))
    v = estado.get("v", np.zeros_like(theta))
    # TU CÓDIGO (no olvides guardar t, m y v en estado)
    return None

In [ ]:
# Comprobación con los ejemplos resueltos del documento
def correr_1d(paso, grads=None, n=3, lr=0.1):
    th, est, out = np.array([1.0]), {}, []
    for k in range(n):
        g = th.copy() if grads is None else np.array([grads[k]])   # f = x²/2 -> g = x
        th = paso(th, g, est, lr); out.append(float(th[0]))
    return out

assert np.allclose(correr_1d(paso_momentum), [0.9, 0.72, 0.486]), "Momentum: deberían salir 0,9; 0,72; 0,486"
th_adam = correr_1d(paso_adam, grads=[0.5, -0.1], n=2, lr=0.01)
assert np.allclose(th_adam, [0.99, 0.98489], atol=1e-5), f"Adam: esperaba 0,99 y 0,98489, salen {th_adam}"
th_rms = correr_1d(paso_rmsprop, grads=[0.5, 0.5], n=2, lr=0.01)
assert np.allclose(th_rms, [1 - 0.01/np.sqrt(0.1), 1 - 0.01/np.sqrt(0.1) - 0.01/np.sqrt(0.19)]), "RMSProp: revisa la media móvil de g²"
print("Correcto. Fíjate: el primer paso de Adam mide exactamente lr = 0,01, sea cual sea la escala de g.")

## Parte 4 · Carrera en el valle

Todos salen de $(-1{,}5;\ 2)$ y dan 5.000 pasos. Las tasas de aprendizaje están ajustadas a mano: el descenso de gradiente diverge con $\eta = 0{,}003$.

In [ ]:
def optimizar(paso, lr, n=5000, p0=(-1.5, 2.0), **hp):
    th, est = np.array(p0, float), {}
    tray = [th.copy()]
    for k in range(n):
        lr_k = lr(k) if callable(lr) else lr
        th = paso(th, rosen_grad(th), est, lr_k, **hp)
        tray.append(th.copy())
    return np.array(tray)

config = {
    "GD (η=1e-3)":       (paso_gd, 1e-3),
    "Momentum (η=1e-3)": (paso_momentum, 1e-3),
    "RMSProp (η=1e-3)":  (paso_rmsprop, 1e-3),
    "Adam (η=2e-2)":     (paso_adam, 2e-2),
}
tray = {nombre: optimizar(p, lr) for nombre, (p, lr) in config.items()}
for nombre, T in tray.items():
    fs = np.array([rosen(p) for p in T])
    llega = np.argmax(fs < 1e-4) if (fs < 1e-4).any() else None
    print(f"{nombre:<20} final {T[-1].round(4)}  f = {fs[-1]:.2e}  primer paso con f<1e-4: {llega}")

In [ ]:
xs, ys = np.linspace(-2, 2, 400), np.linspace(-1, 3, 400)
XX, YY = np.meshgrid(xs, ys)
ZZ = (1 - XX)**2 + 100 * (YY - XX**2)**2

fig, axs = plt.subplots(2, 2, figsize=(11, 9), sharex=True, sharey=True)
for ax, (nombre, T) in zip(axs.flat, tray.items()):
    ax.contour(XX, YY, ZZ, levels=np.logspace(-1, 3.5, 18), cmap="Greys", linewidths=0.6)
    ax.plot(T[:, 0], T[:, 1], lw=1.2)
    ax.plot(T[::100, 0], T[::100, 1], "o", ms=3)     # un punto cada 100 pasos: separación = velocidad
    ax.plot(1, 1, "k*", ms=14); ax.plot(-1.5, 2, "ko")
    ax.set_xlim(-2, 2); ax.set_ylim(-1, 3); ax.set_title(nombre)
plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 4))
for nombre, T in tray.items():
    plt.semilogy([rosen(p) + 1e-30 for p in T], label=nombre)
plt.xlabel("paso"); plt.ylabel("f"); plt.title("Pérdida (escala log)"); plt.legend(); plt.show()

**Preguntas.** (a) ¿Por qué el descenso de gradiente cae enseguida al valle y luego avanza tan despacio? (b) ¿Qué hace el momentum al llegar al fondo del valle? (c) RMSProp y Adam normalizan cada coordenada: ¿se nota en la forma de su trayectoria?

## Parte 5 · Warmup y cosine decay

El calendario típico de preentrenamiento sube lineal durante $W$ pasos y luego baja como un coseno hasta $\eta_{\min}$:

$$\eta(t) = \begin{cases} \eta_{\max}\, t/W & t < W \\ \eta_{\min} + \tfrac12(\eta_{\max}-\eta_{\min})\big(1 + \cos(\pi\,\tfrac{t-W}{T-W})\big) & W \le t \le T \end{cases}$$

**Ejercicio 5.** Implementa `lr_cosine(t, peak, warmup, total, lr_min)`.

In [ ]:
def lr_cosine(t, peak, warmup, total, lr_min):
    # TU CÓDIGO
    return None

In [ ]:
# Comprobación con el ejercicio 10 del documento
args = dict(peak=6e-4, warmup=1000, total=11000, lr_min=6e-5)
for t, esperado in [(0, 0), (500, 3e-4), (1000, 6e-4), (6000, 3.3e-4), (8500, 1.3908e-4), (11000, 6e-5)]:
    assert abs(lr_cosine(t, **args) - esperado) < 1e-8, f"t = {t}: esperaba {esperado}"
ts = np.arange(11001)
plt.figure(figsize=(7, 3)); plt.plot(ts, [lr_cosine(t, **args) for t in ts])
plt.xlabel("paso"); plt.ylabel("η"); plt.title("Warmup lineal + cosine decay"); plt.show()
print("Correcto")

**Por qué bajar η al final.** Con SGD el ruido del minibatch impide asentarse en el mínimo: el parámetro «tiembla» con amplitud proporcional a η. Compruébalo con la regresión de la parte 2 (B = 10, 1.000 pasos).

In [ ]:
A = np.c_[X, np.ones(N)]
theta_opt = np.linalg.lstsq(A, Y, rcond=None)[0]          # mínimo exacto
perd = lambda th: 0.5 * np.mean((th[0] * X + th[1] - Y)**2)

def sgd(lr_fn, pasos=1000, B=10):
    r, th = np.random.default_rng(1), np.zeros(2)
    for k in range(pasos):
        th = th - lr_fn(k) * grad_mb(th, r.choice(N, B, replace=False))
    return perd(th) - perd(theta_opt)

exceso_fijo = sgd(lambda k: 0.1)
exceso_cos = sgd(lambda k: lr_cosine(k, 0.1, 50, 1000, 1e-3))
print(f"Exceso de pérdida final: η fijo 0,1 -> {exceso_fijo:.2e} · warmup + coseno -> {exceso_cos:.2e}")

## Parte 6 · AdamW: weight decay desacoplado

Adam + L2 suma $\lambda\theta$ al gradiente **antes** de normalizar; AdamW resta $\eta\lambda\theta$ **después**:

$$\theta \leftarrow \theta - \eta\Big(\frac{\hat m}{\sqrt{\hat v}+\varepsilon} + \lambda\,\theta\Big)$$

**Ejercicio 6.** Implementa `paso_adamw` reutilizando `paso_adam` (pista: llama a `paso_adam` y resta después el término de decaimiento).

In [ ]:
def paso_adamw(theta, g, estado, lr, wd=0.1, **kw):
    # TU CÓDIGO
    return None

def paso_adam_l2(theta, g, estado, lr, wd=0.1, **kw):
    return paso_adam(theta, g + wd * theta, estado, lr, **kw)

In [ ]:
# Comprobación: primer paso con g = 0,5, theta = 1, lr = 0,01, wd = 0,1
a = paso_adamw(np.array([1.0]), np.array([0.5]), {}, 0.01, wd=0.1)
b = paso_adam_l2(np.array([1.0]), np.array([0.5]), {}, 0.01, wd=0.1)
assert np.allclose(a, 0.989), f"AdamW debería dar 0,989 (sale {a})"
assert np.allclose(b, 0.99), "Adam + L2 da 0,99: la normalización se 'come' el término λθ"
assert np.allclose(paso_adamw(np.array([1.0]), np.array([0.5]), {}, 0.01, wd=0.0), 0.99), "Con wd = 0 debe coincidir con Adam"
print("Correcto")

In [ ]:
# Cuánto encoge cada peso por paso según su historial de gradientes (v̂)
lr, wd, w = 1e-3, 0.1, 1.0
for v_hat in [1e-4, 1.0]:
    print(f"v̂ = {v_hat:<6}  Adam+L2: {lr*wd*w/np.sqrt(v_hat):.4f}   AdamW: {lr*wd*w:.4f}")

## Parte 7 · Retos

1. **Momentum de Nesterov.** Evalúa el gradiente en $\theta - \eta\beta v$ y compáralo con la bola pesada en Rosenbrock.
2. **Mapa de estabilidad.** Para el descenso de gradiente, barre $\eta \in [10^{-4}, 4\cdot10^{-3}]$ y dibuja la pérdida final. ¿Dónde diverge? Compáralo con $2/\lambda_{\max}$.
3. **SGD ruidoso.** Añade ruido gaussiano al gradiente de Rosenbrock (desviación 10) y compara GD y Adam. ¿Qué optimizador sufre más?
4. **Sensibilidad de Adam.** Repite la carrera con $\beta_2 = 0{,}95$ (valor de GPT-3 y LLaMA). ¿Cambia algo?

---
### Soluciones de referencia
Inténtalo antes de mirar.

In [ ]:
# Ejercicio 1
#     return np.array([-2*(1 - x) - 400*x*(y - x**2), 200*(y - x**2)])
# Ejercicio 2
#     for k in range(1, max_it + 1):
#         theta = theta - eta * H @ theta
#         if np.linalg.norm(theta) < tol * n0:
#             return k
#     return max_it
# Ejercicio 3
#     r = w * X[idx] + b - Y[idx]
#     return np.array([np.mean(r * X[idx]), np.mean(r)])
# Ejercicio 4
#   momentum:  v = beta * v + g;  estado["v"] = v;  return theta - lr * v
#   rmsprop:   s = rho * s + (1 - rho) * g**2;  estado["s"] = s;  return theta - lr * g / (np.sqrt(s) + eps)
#   adam:      m = b1 * m + (1 - b1) * g;  v = b2 * v + (1 - b2) * g**2
#              estado.update(t=t, m=m, v=v)
#              m_hat = m / (1 - b1**t);  v_hat = v / (1 - b2**t)
#              return theta - lr * m_hat / (np.sqrt(v_hat) + eps)
# Ejercicio 5
#     if t < warmup:
#         return peak * t / warmup
#     p = (t - warmup) / (total - warmup)
#     return lr_min + 0.5 * (peak - lr_min) * (1 + np.cos(np.pi * p))
# Ejercicio 6
#     return paso_adam(theta, g, estado, lr, **kw) - lr * wd * theta